# Chroma — Collections, Ingest, and Query

**Chroma** is an open-source embedding database designed for developer ergonomics: local-first, Python-native, and easy to embed in RAG prototypes. It is listed in this repo's `requirements.txt` (`chromadb`, `langchain-chroma`) and is the hands-on vector store for this curriculum.

This notebook covers Chroma's client and collection model, ingesting documents with **explicit OpenAI embeddings**, querying by `query_embeddings`, reading distances and metadata, `get`/`peek`/`count`, persistence with `PersistentClient`, and a minimal end-to-end ingest→search flow aligned with production RAG patterns.

Prerequisites: **01–05** in this section. Next: **07. Metadata Filters and Index Management**.

In [ ]:
import chromadb
import numpy as np

np.set_printoptions(precision=4, suppress=True)

---

## 1. Why Chroma in This Curriculum

| Advantage | Detail |
|-----------|--------|
| **Local dev** | No cloud account required for vector storage |
| **Simple API** | `create_collection`, `add`, `query` |
| **Python-first** | Matches FastAPI / notebook workflows |
| **Persistence** | `PersistentClient` writes to disk |
| **Ecosystem** | LangChain integration via `langchain-chroma` |

Chroma is not the only choice for production (see notebook 08) — but it is the best place to **learn vector DB mechanics** before operating larger systems.

---

## 2. Core Concepts

| Concept | Description |
|---------|-------------|
| **Client** | Factory for collections (`Client()` or `PersistentClient(path)`) |
| **Collection** | Named index — vectors + optional documents + metadata |
| **add** | Insert or upsert by `ids` |
| **query** | Similarity search — `query_embeddings` or `query_texts` |
| **get** | Fetch by ids or filters |
| **delete** | Remove by ids or `where` filter |

### 2.1 Client Types

| Client | Data lifetime |
|--------|---------------|
| `chromadb.Client()` | In-memory — **lost** when process exits |
| `chromadb.PersistentClient(path="./chroma_data")` | Survives restarts |

### 2.2 Who Embeds the Text?

| Mode | How |
|------|-----|
| **Chroma embedding function** | Pass `documents` only; Chroma calls a default model |
| **Bring your own vectors** (recommended here) | You call OpenAI; pass `embeddings` to `add` and `query_embeddings` to `query` |

Explicit OpenAI embeddings keep behavior aligned with **03. Embedding APIs** and production RAG.

---

## 3. Collection Configuration

When creating a collection you can specify metadata and distance function:

```python
col = client.create_collection(
    name="docs_v1",
    metadata={"hnsw:space": "cosine"},  # cosine | l2 | ip
)
```

| `hnsw:space` | Maps to |
|--------------|--------|
| `cosine` | Cosine distance (common for text) |
| `l2` | Euclidean |
| `ip` | Inner product (use with normalized vectors) |

Match the metric to your embedding model guidance (notebook 02).

---

## 4. The `add` API

```python
collection.add(
    ids=["chunk-1", "chunk-2"],
    documents=["text a", "text b"],
    embeddings=[[...], [...]],
    metadatas=[{"source": "a.md"}, {"source": "b.md"}],
)
```

| Rule | Reason |
|------|--------|
| **Ids unique** per collection | Upsert semantics — same id overwrites |
| **Parallel lists same length** | ids, documents, embeddings, metadatas align by index |
| **Embedding dim constant** | All vectors same length $d$ |

Batch large ingests — do not call `add` once per chunk in a tight loop if you can batch hundreds per call.

---

## 5. The `query` API

```python
results = collection.query(
    query_embeddings=[query_vector],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)
```

Response shape (batch query returns list per query):

```python
results["ids"][0]         # top ids for first query
results["documents"][0] # matching text
results["distances"][0]   # lower = closer (for distance metrics)
results["metadatas"][0]
```

**Distance vs similarity:** Chroma returns **distances**. For cosine space, related texts have **smaller** distance. Do not confuse with cosine similarity scores from NumPy (higher = closer).

---

## 6. Demonstration Setup

Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"

DOCUMENTS = [
    {"id": "1", "text": "Our Notes API stores notes in memory and exposes REST endpoints with FastAPI.", "source": "api-docs"},
    {"id": "2", "text": "Alembic manages database schema migrations for SQLAlchemy projects.", "source": "db-docs"},
    {"id": "3", "text": "JWT tokens are used for stateless authentication in APIs.", "source": "security-docs"},
    {"id": "4", "text": "Pytest fixtures simplify database setup in integration tests.", "source": "test-docs"},
]


def embed_texts(texts: list[str]) -> list[list[float]]:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(r.data, key=lambda x: x.index)
    return [d.embedding for d in ordered]


def fresh_collection(name: str = "lesson_collection"):
    """In-memory client; delete/recreate for repeatable notebook runs."""
    client = chromadb.Client()
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name=name, metadata={"hnsw:space": "cosine"})


print("Setup OK")

---

## 7. Demonstration: Create Collection and Ingest

In [ ]:
col = fresh_collection()
texts = [d["text"] for d in DOCUMENTS]
embeddings = embed_texts(texts)

col.add(
    ids=[d["id"] for d in DOCUMENTS],
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"source": d["source"]} for d in DOCUMENTS],
)

print(f"Collection: {col.name}")
print(f"Count: {col.count()}")
print(f"Embedding dim: {len(embeddings[0])}")

---

## 8. Demonstration: Query with `query_embeddings`

In [ ]:
questions = [
    "What tool handles SQL schema migrations?",
    "How does API authentication work?",
]

for question in questions:
    q_emb = embed_texts([question])
    results = col.query(
        query_embeddings=q_emb,
        n_results=2,
        include=["documents", "metadatas", "distances"],
    )
    print(f"Q: {question}")
    for doc_id, doc, meta, dist in zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        print(f"  distance={dist:.4f}  id={doc_id}  source={meta['source']}")
        print(f"    {doc}")
    print()

Expect migration questions to rank **id=2** (Alembic) first; auth questions rank **id=3** (JWT).

---

## 9. Demonstration: `get`, `peek`, and `count`

In [ ]:
print("count():", col.count())

got = col.get(ids=["2"], include=["documents", "metadatas"])
print("\nget(id='2'):")
print("  doc:", got["documents"][0])
print("  meta:", got["metadatas"][0])

peek = col.peek(limit=2)
print("\npeek(limit=2) ids:", peek["ids"])

---

## 10. Demonstration: Incremental `add` (Upsert)

In [ ]:
new_chunk = {
    "id": "5",
    "text": "Run alembic upgrade head in CI before deploying.",
    "source": "db-docs",
}
col.add(
    ids=[new_chunk["id"]],
    documents=[new_chunk["text"]],
    embeddings=embed_texts([new_chunk["text"]]),
    metadatas=[{"source": new_chunk["source"]}],
)
print("Count after add id=5:", col.count())

# Upsert — same id "2", new text
updated = "Alembic autogenerate compares models to the live database schema."
col.add(
    ids=["2"],
    documents=[updated],
    embeddings=embed_texts([updated]),
    metadatas=[{"source": "db-docs"}],
)
print("After upsert id=2:", col.get(ids=["2"])["documents"][0])

---

## 11. `query_texts` vs Bring-Your-Own Embeddings

If the collection is created **with** an embedding function, you can query with raw text:

```python
results = col.query(query_texts=["migration tool"], n_results=2)
```

In this curriculum we pass **`query_embeddings`** from OpenAI so:

- Index and query use the **same** model
- Behavior matches production FastAPI services
- You are not surprised by Chroma's default embedder differing from ingest

---

## 12. Persistence with `PersistentClient`

In [ ]:
import tempfile
from pathlib import Path

persist_dir = Path(tempfile.mkdtemp(prefix="chroma_lesson_"))
print("Persist path:", persist_dir)

pclient = chromadb.PersistentClient(path=str(persist_dir))
pcol = pclient.get_or_create_collection("persisted_docs", metadata={"hnsw:space": "cosine"})

if pcol.count() == 0:
    pcol.add(
        ids=["p1"],
        documents=["Persistent storage survives kernel restarts."],
        embeddings=embed_texts(["Persistent storage survives kernel restarts."]),
        metadatas=[{"source": "demo"}],
    )

print("Persistent collection count:", pcol.count())

# Simulate new process — new client, same path
pclient2 = chromadb.PersistentClient(path=str(persist_dir))
pcol2 = pclient2.get_collection("persisted_docs")
print("Reopened count:", pcol2.count())
print("Document:", pcol2.get(ids=["p1"])["documents"][0])

In projects, use a fixed path like `./data/chroma` under your app root — not a temp dir.

---

## 13. Ingest Helper Pattern

Reusable function — chunk records from notebook 04 → Chroma:

In [ ]:
def ingest_records(collection, records: list[dict], batch_size: int = 50) -> int:
    """records: [{id, text, metadata}]"""
    total = 0
    for i in range(0, len(records), batch_size):
        batch = records[i : i + batch_size]
        texts = [r["text"] for r in batch]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=texts,
            embeddings=embed_texts(texts),
            metadatas=[r["metadata"] for r in batch],
        )
        total += len(batch)
    return total


demo_col = fresh_collection("ingest_helper_demo")
records = [
    {"id": f"r{i}", "text": d["text"], "metadata": {"source": d["source"]}}
    for i, d in enumerate(DOCUMENTS)
]
n = ingest_records(demo_col, records)
print(f"Ingested {n} records; count={demo_col.count()}")

---

## 14. Wiring to RAG (Preview)

Retrieval-only step — generation comes in **08. RAG Fundamentals**:

```python
def retrieve(question: str, k: int = 3) -> list[str]:
    q_emb = embed_texts([question])
    hits = col.query(query_embeddings=q_emb, n_results=k, include=["documents"])
    return hits["documents"][0]

context = "\n".join(retrieve("How do migrations work?"))
# → pass context + question to client.chat.completions.create(...)
```

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| In-memory client in prod | Data loss | `PersistentClient` |
| Default Chroma embedder at query, OpenAI at ingest | Wrong rankings | `query_embeddings` |
| Mismatched embedding dims | Runtime error | Validate `len(embedding)` |
| One id per chunk not stable | Duplicates on re-ingest | Deterministic chunk ids |
| Ignoring distance scale | Bad thresholds | Calibrate on eval set |
| `add` one row at a time in loop | Slow ingest | Batch + `ingest_records` |

---

## 16. Summary

Chroma **collections** store ids, embeddings, documents, and metadata. Use **`add`** to ingest (upsert by id), **`query`** with **`query_embeddings`** when using OpenAI, and **`PersistentClient`** for durability.

Understand **distances** (lower = closer) vs NumPy **similarity** (higher = closer). Batch ingest; match cosine metric to your embedding workflow.

Next: **07. Metadata Filters and Index Management** — `where` filters, deletes, and re-indexing.